## Subagents pattern

In [10]:
from __future__ import annotations

import argparse
import operator
import os
import time
from typing import Annotated, TypedDict

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.runnables import RunnableLambda
from langgraph.graph import END, START, StateGraph
from langgraph.types import Send
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_tavily import TavilySearch

from dotenv import load_dotenv


In [11]:
load_dotenv(override=True)

True

In [19]:
model = ChatOpenAI(
    model=os.getenv("CHAT_MODEL_DEPLOYMENT_NAME"),
    base_url=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY")
)

In [20]:
search_tool = TavilySearch(
    max_results=10,
    topic="general",
    search_depth = "advanced"
)

# Create a subagent
research_agent = create_agent(
    model=model, 
    tools=[search_tool]
)


In [21]:
writer_agent = create_agent(
    model=model,
    system_prompt="You are a writing specialist..."
)


In [22]:
# Registry of available sub-agents
SUBAGENTS = {
    "research": research_agent,
    "writer": writer_agent,
}


@tool
def task(
    agent_name: str,
    description: str
) -> str:
    """Launch an ephemeral subagent for a task.

    Available agents:
    - research: Research and fact-finding
    - writer: Content creation and editing
    """
    agent = SUBAGENTS[agent_name]
    result = agent.invoke({
        "messages": [
            {"role": "user", "content": description}
        ]
    })
    return result["messages"][-1].content


In [23]:
main_agent = create_agent(
    model=model,
    tools=[task],
    system_prompt=(
        "You coordinate specialized sub-agents. "
        "Available: research (fact-finding), "
        "writer (content creation). "
        "Use the task tool to delegate work."
    ),
)

In [24]:
response = main_agent.invoke({
    "messages": [
        {"role": "user", "content": "Please research the topic of AI advancements and then write a summary."}
    ]
})

print(response["messages"][-1].content)

## Summary of AI Advancements

Artificial intelligence has advanced rapidly, particularly in reasoning, multimodal interaction, software development, scientific research, and automation. However, progress remains uneven: AI systems can perform impressive tasks but still make confident mistakes and require human oversight in many real-world applications.

A major development has been the emergence of **reasoning models**. Systems such as OpenAI’s o1 and o3, DeepSeek-R1, and Google’s Gemini 2.5 are designed to spend more computing time planning and checking their answers. They have achieved strong results in areas such as mathematics, coding, and scientific problem-solving. Their limitations include higher cost, slower responses, and unreliable performance when problems differ from benchmark tests.

AI has also become increasingly **multimodal**, meaning it can process and generate text, images, audio, and video. Models such as GPT-4o support more natural voice and visual conversations, 

In [25]:
response = main_agent.invoke({
    "messages": [
        {"role": "user", "content": "Write a summary of AI advancements."}
    ]
})

print(response["messages"][-1].content)

## Summary of AI Advancements

Artificial intelligence has advanced rapidly in recent years, especially through the development of increasingly capable foundation models. These systems can generate and analyze text, write code, understand images and audio, summarize large documents, and solve some complex, multi-step problems.

A major development has been **multimodal AI**, which combines language, vision, speech, video, and code in a single system. This has enabled more natural voice assistants, image and document analysis, real-time translation, and increasingly sophisticated creative tools for generating images, music, and video.

AI is also becoming more **agentic**. Rather than simply answering questions, AI agents can search for information, use software tools, browse websites, write and test code, and complete parts of larger workflows. These systems are increasingly used in software development, customer service, research, and business operations, although they can still make 